# Intents, Slots and Wrong Transcripts [Agent Patterns - Module 11]

> **MLCourse - Agentic AI - Agent Patterns**

A voice agent that only chats is easy. A voice agent that **does something** -
reschedules a delivery, refunds an order, books an appointment - has to pull
structured values out of a transcript it cannot fully trust.

That combination is the whole difficulty of voice agents:

- speech carries the values agents need (order numbers, dates, amounts),
- and those are exactly the tokens STT gets wrong most often.

### What you will learn

1. Intent + slot extraction from a transcript, with a pinned schema.
2. Normalising spoken numbers and dates ("four four seven one" -> 4471).
3. Detecting missing slots and asking one clarifying question.
4. The confirmation pattern: read critical values back before acting.

### Key takeaways

- Extract into a schema; never let free text become a tool argument.
- Spoken numbers arrive as words. Normalise deliberately.
- Confirm anything expensive or irreversible. Voice is too lossy not to.

### Setup: imports, environment, track discovery


In [ ]:
import os
import re
import sys
import json
import time
import wave
import random
from pathlib import Path
from dotenv import load_dotenv

def _find_track(depth=6):
    p = Path.cwd()
    for _ in range(depth):
        if (p / "03_agentic_ai").is_dir():
            return p
        p = p.parent
    return Path.cwd()

TRACK = _find_track()
# The .env lives INSIDE the track folder, not the repo root that
# _find_track() returns - joining ".env" onto TRACK alone is a silent no-op.
load_dotenv(TRACK / "03_agentic_ai" / ".env", override=False)

GROQ_API_KEY = os.environ["GROQ_API_KEY"]   # loud failure if missing, by design
CHAT_MODEL = "qwen/qwen3.8-27b"              # Groq-hosted; never OpenAI
STT_MODEL = "whisper-large-v3"               # Groq-hosted speech-to-text

HERE = Path.cwd().resolve()
AUDIO = HERE / "audio"
AUDIO.mkdir(exist_ok=True)

print(f"Track root : {TRACK}")
print(f"Audio dir  : {AUDIO}")
print(f"Chat model : {CHAT_MODEL}")
print(f"STT model  : {STT_MODEL}")


### Making audio without a microphone


In [ ]:
# This module must run headless, so there is no mic. We SYNTHESISE the input
# audio with pyttsx3, which drives the operating system's built-in TTS voice
# (SAPI5 on Windows, NSSpeechSynthesizer on macOS, espeak on Linux).
#
# It is fully offline, needs no key, and gives us a real .wav file with real
# speech in it - which is exactly what the STT step needs.

import pyttsx3

def speak_to_file(text, path, rate=150):
    """Render `text` to a .wav file using the OS voice. Returns the Path."""
    path = Path(path)
    engine = pyttsx3.init()
    engine.setProperty("rate", rate)          # words per minute
    engine.save_to_file(text, str(path))
    engine.runAndWait()
    engine.stop()
    return path

def wav_info(path):
    with wave.open(str(path)) as w:
        return {
            "seconds": round(w.getnframes() / w.getframerate(), 2),
            "sample_rate": w.getframerate(),
            "channels": w.getnchannels(),
            "bytes": Path(path).stat().st_size,
        }

print("speak_to_file() ready (offline OS voice)")


### Groq speech-to-text


In [ ]:
# whisper-large-v3 on Groq. Multipart upload: the file goes in `files`, the
# parameters go in `data`. Backoff included - the free tier is shared.

import requests

STT_URL = "https://api.groq.com/openai/v1/audio/transcriptions"

def transcribe(path, response_format="json", language="en"):
    """Send a .wav to Groq whisper-large-v3 and return the parsed response."""
    for attempt in range(5):
        with open(path, "rb") as fh:
            r = requests.post(
                STT_URL,
                headers={"Authorization": f"Bearer {GROQ_API_KEY}"},
                files={"file": (Path(path).name, fh, "audio/wav")},
                data={"model": STT_MODEL,
                      "response_format": response_format,
                      "language": language,
                      "temperature": 0},
                timeout=120,
            )
        if r.status_code == 200:
            return r.json()
        wait = 2 ** attempt + random.random()
        print(f"  HTTP {r.status_code}, retry {attempt+1} in {wait:.1f}s")
        time.sleep(wait)
    raise RuntimeError(f"STT failed: {r.status_code} {r.text[:300]}")

print("transcribe() ready")


### Groq chat, with 429 backoff


In [ ]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

def ask(prompt, system="You are a precise assistant.", max_tokens=500, temperature=0.0):
    for attempt in range(5):
        try:
            r = client.chat.completions.create(
                model=CHAT_MODEL,
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": prompt}],
                temperature=temperature, max_tokens=max_tokens,
            )
            return r.choices[0].message.content
        except Exception as e:
            wait = 2 ** attempt + random.random()
            print(f"  retry {attempt+1} in {wait:.1f}s ({type(e).__name__})")
            time.sleep(wait)
    raise RuntimeError("Groq chat failed after 5 attempts")

print("ask() ready")


### 1. From transcript to structure

The agent's job here is not to answer; it is to fill a form. We declare the
form, and the model fills it - the same discipline as the browser module: a
pinned schema, `null` for absent values, and Python validation afterwards.

The critical rule for voice: **`null` when unsure, never a guess.** A wrong
order number silently reschedules someone else's delivery. A missing one just
prompts a question.

### Generate a caller utterance and transcribe it


In [ ]:
SCRIPT = ("Hi, this is Anita. I need to reschedule the delivery for order "
          "four four seven one to Friday morning please.")

wav = speak_to_file(SCRIPT, AUDIO / "reschedule.wav")
transcript = transcribe(wav)["text"].strip()

print("script    :", SCRIPT)
print("transcript:", transcript)


### Intent + slots


In [ ]:
INTENTS = ["reschedule_delivery", "check_status", "cancel_order",
           "report_damage", "other"]

SLOT_PROMPT = """A caller said the following to a delivery company.
Return ONLY a JSON object with exactly these keys:

  "intent"       : one of {intents}
  "caller_name"  : string or null
  "order_number" : string of DIGITS ONLY, or null if not clearly stated
  "when"         : the requested time as the caller said it, or null
  "confidence"   : "high" or "low" - low if the audio transcript looks garbled

Rules:
- Spoken digits must be joined: "four four seven one" -> "4471".
- Use null rather than guessing. A wrong order number is worse than a missing one.
- No prose, no markdown fence.

TRANSCRIPT:
{t}"""

raw = ask(SLOT_PROMPT.format(intents=INTENTS, t=transcript),
          system="You extract structured data. Output JSON only.")
print(raw)


### Parse and validate


In [ ]:
def parse_json(text):
    text = text.strip()
    fence = re.search(r"```(?:json)?\s*(.*?)```", text, re.S)
    if fence:
        text = fence.group(1).strip()
    s, e = text.find("{"), text.rfind("}")
    if s == -1:
        raise ValueError(f"no JSON in: {text[:200]}")
    return json.loads(text[s:e + 1])

REQUIRED = {"intent", "caller_name", "order_number", "when", "confidence"}

slots = parse_json(raw)
assert REQUIRED <= set(slots), f"missing keys: {REQUIRED - set(slots)}"
assert slots["intent"] in INTENTS, f"unknown intent: {slots['intent']}"

# order_number must be digits or null - the model does not get to be creative
if slots["order_number"] is not None:
    assert re.fullmatch(r"\d+", str(slots["order_number"])), \
        f"order_number is not digits: {slots['order_number']!r}"

print("validated slots:")
for k, v in slots.items():
    print(f"  {k:14s} {v!r}")


### 2. When the transcript is damaged

The last example was clean. Real calls are not: phone codecs, background
noise, accents, people talking over each other. What happens when the
critical value is mangled?

We can simulate that honestly by corrupting the transcript the way ASR
actually corrupts it - a plausible-sounding wrong word - and watching what
the extractor does with it.

### A garbled transcript


In [ ]:
GARBLED = ("Hi this is Anita I need to reschedule the delivery for order "
           "for for seven what to Friday morning please")

raw2 = ask(SLOT_PROMPT.format(intents=INTENTS, t=GARBLED),
           system="You extract structured data. Output JSON only.")
slots2 = parse_json(raw2)

print("garbled transcript:", GARBLED)
print()
for k, v in slots2.items():
    print(f"  {k:14s} {v!r}")
print()
if slots2["order_number"] in (None, ""):
    print("Good: the extractor declined to guess.")
else:
    print(f"The extractor produced {slots2['order_number']!r} anyway.")
    print("This is the dangerous case, and it is why the schema alone is not")
    print("enough - the confirmation step in section 4 is what saves you.")


### 3. Asking exactly one question

When a required slot is missing, the agent must ask - but a voice caller can
only hold one question in their head at a time. "Can I get your order number,
the date, and your postcode?" produces an answer to one of the three.

So: find the missing slots, pick the **most important one**, ask about that,
and loop.

### Slot filling policy


In [ ]:
# Ordered by importance: ask about the first missing one.
REQUIRED_FOR = {
    "reschedule_delivery": ["order_number", "when"],
    "check_status":        ["order_number"],
    "cancel_order":        ["order_number"],
    "report_damage":       ["order_number"],
    "other":               [],
}

def next_missing(slots):
    for slot in REQUIRED_FOR.get(slots["intent"], []):
        if slots.get(slot) in (None, "", "null"):
            return slot
    return None

for label, s in [("clean", slots), ("garbled", slots2)]:
    m = next_missing(s)
    print(f"{label:8s} intent={s['intent']:20s} missing -> {m or 'nothing, ready to act'}")


### Generate the one question


In [ ]:
missing = next_missing(slots2)

QUESTION_PROMPT = ("The caller wants to {intent}. You still need their "
                   "'{slot}'. Ask for it in ONE short spoken sentence. "
                   "No greeting, no markdown.")

if missing:
    q = ask(QUESTION_PROMPT.format(intent=slots2["intent"].replace("_", " "),
                                   slot=missing.replace("_", " ")),
            system="You are a phone agent. One short sentence only.",
            max_tokens=80).strip()
    print("agent asks:", q)
else:
    print("nothing missing - no question needed")


### 4. The confirmation pattern

This is the part that makes voice agents safe, and it is not optional.

Before any action that costs money, changes a booking, or cannot be undone,
the agent **reads the critical values back** and waits for a yes. Not a
summary of its intentions - the literal values, spoken as digits.

Why it works: the caller knows their own order number. They cannot audit the
agent's reasoning, but they can absolutely catch "four four seven **two**".

### Build the confirmation utterance


In [ ]:
def spell_digits(s):
    """4471 -> 'four four seven one' - far more robust through a synthesiser."""
    words = {"0": "zero", "1": "one", "2": "two", "3": "three", "4": "four",
             "5": "five", "6": "six", "7": "seven", "8": "eight", "9": "nine"}
    return " ".join(words[c] for c in str(s) if c in words)

def confirmation_text(slots):
    return (f"Just to confirm: order number "
            f"{spell_digits(slots['order_number'])}, "
            f"rescheduled to {slots['when']}. Is that correct?")

conf = confirmation_text(slots)
print(conf)

conf_wav = speak_to_file(conf, AUDIO / "confirm.wav")
print("\nspoken to:", conf_wav.name, wav_info(conf_wav))


### The gate: nothing executes without an explicit yes


In [ ]:
def caller_said_yes(reply_text):
    """Deterministic. Do NOT ask an LLM whether the caller consented."""
    t = reply_text.lower()
    yes = any(w in t for w in ("yes", "yeah", "yep", "correct", "that's right",
                              "that is right", "confirm"))
    no = any(w in t for w in ("no", "nope", "wrong", "incorrect", "not right"))
    return yes and not no

def execute_reschedule(order_number, when):
    return f"[ACTION] order {order_number} rescheduled to {when}"

for caller_reply in ["Yes that's right", "No, that's wrong", "erm, hold on"]:
    approved = caller_said_yes(caller_reply)
    outcome = (execute_reschedule(slots["order_number"], slots["when"])
               if approved else "[NO ACTION] re-ask or escalate")
    print(f"  caller: {caller_reply!r:24s} -> {outcome}")


### Why `caller_said_yes` is plain Python

Consent is a control-flow decision, so it belongs in code you can read and
test - not in a model call that a confusing transcript could flip. The same
principle as the browser module: the model proposes, deterministic code
decides.

(A real system needs more than substring matching - "no, yes go ahead" breaks
it. The point is *where* the decision lives, not that this particular
matcher is production grade.)

### Pitfalls recap

- **Letting the model guess an ID.** `null` is a question; a wrong number is
  an incident.
- **Asking for several slots at once.** Callers answer one. Ask one.
- **Confirming a paraphrase.** Read back the literal values, and spell digits
  out - synthesisers mangle "4471" far more often than "four four seven one".
- **Using an LLM as the consent gate.** Deterministic code, always.
- **Skipping confirmation because the transcript "looked fine".** Whether it
  was fine is exactly what you cannot know.

### Next

Notebook 04 collects the operational realities: cost, chunking, streaming,
and when voice is the wrong interface.